# T10-bonus · App and flows as code

## Goal

Wire the Git-Integration-synced canvas app source and the unpacked flow
JSON into the same CI/CD pipeline `T9-bonus` built for the agent fleet —
one promotion gate covering the agent, the app, and its flows together.


## Prereqs

Asserted below, not just stated — this cell fails loudly if a prior notebook's step wasn't actually completed.


In [ ]:
from pathlib import Path
assert Path("../apps/renewal-desk-canvas/src/Screens/SupplierDetail.pa.yaml").exists(), "run 26-31 first"
assert Path("../infra/pipelines/deploy.yml").exists(), "run T9-bonus first"


## Concept

`T9-bonus` gated promotion on `run_suite()` against the agent alone. Once
an app and its flows depend on that same agent's contract (`30`'s
prompt-shape, `31`'s decision-field write-back), a promotion that only
checks the agent is an incomplete gate — the app could still break on a
change the agent-only suite doesn't exercise. This notebook extends the
existing pipeline rather than building a parallel one: the build stage
now also exports+unpacks the app's solution (no auth needed, same as
`pac copilot pack`), and the deploy gate adds the `app-integration`-tagged
cases to what it checks before `publish-changes`.


## Build


In [ ]:
from pathlib import Path
agent_ci = Path("../infra/pipelines/agent-ci.yml")
text = agent_ci.read_text()
app_build_step = '''
      - name: Export + unpack renewal-desk-canvas app and flows
        run: |
          pac solution export --name crd-renewal-desk-flows --path dist/renewal-desk-flows.zip --managed false
          pac solution unpack --zipfile dist/renewal-desk-flows.zip --folder apps/renewal-desk-canvas/flows/_unpacked --allowWrite true
'''
if "renewal-desk-canvas" not in text:
    text = text.replace(
        '''      - uses: actions/upload-artifact@v4
        with:
          name: crd-solution
          path: dist/crd.zip''',
        app_build_step + '''
      - uses: actions/upload-artifact@v4
        with:
          name: crd-solution
          path: dist/crd.zip

      - uses: actions/upload-artifact@v4
        with:
          name: crd-app-flows
          path: dist/renewal-desk-flows.zip''',
    )
    agent_ci.write_text(text)
print("agent-ci.yml extended to also build the app+flows artifact")


In [ ]:
deploy_yml = Path("../infra/pipelines/deploy.yml")
text = deploy_yml.read_text()
text = text.replace(
    'python -m csx.run_gate --tags core --min-pass-rate 0.8',
    'python -m csx.run_gate --tags core app-integration --min-pass-rate 0.8',
)
deploy_yml.write_text(text)
print("deploy.yml's gate now also covers app-integration-tagged cases")


## Verify

Same harness, same golden set, every notebook.


In [ ]:
with open("../infra/pipelines/deploy.yml") as f:
    text = f.read()
assert "app-integration" in text, "gate extension didn't take"
print("one gate, agent + app, verified")


## Cost


In [ ]:
print("No agent build/publish here — this notebook only edits pipeline YAML. Actual pipeline runs cost what 25's and 31's manual walkthroughs already measured, combined.")


## Teardown


In [ ]:
print("No teardown — this is now the standing promotion gate for the agent fleet and its companion app together.")
